In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns 
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import duckdb
import pandas as pd

In [ ]:
con = duckdb.connect("health_enforcement.duckdb")
df_mart = con.execute("SELECT * FROM enriched_mart").df()

In [ ]:
severity_map = {
    'AA'        : 6.0,  # direct proximate cause of death — $25k-$100k fine
    'A TREBLED' : 4.5,
    'A'         : 4.0,  # imminent danger of death or serious harm
    'B TREBLED' : 3.0,
    'B FIRST'   : 2.5,
    'B'         : 2.0,  # direct relationship to health/safety
    'AP IJ'     : 4.0,  # hospital immediate jeopardy — up to $75k-$125k
    'AP NON-IJ' : 2.5,  # hospital non-IJ — up to $25k
    'AP BR'     : 2.0,
    'AP NHPPD'  : 1.5,  # staffing standard violation
    'FTR AE'    : 2.0,
    'FRTR RES'  : 1.5,
    'FTR BR'    : 1.5,
    'FTR RES'   : 1.5,
    'WF'        : 2.5,  # falsification — integrity violation, bump up
    'WO'        : 2.5,  # omission — same
    'NP'        : 0.5,
    'RD'        : 1.0,
}

# Apply to CLASS_ASSESSED_FINAL
df_mart['FINAL_SEVERITY'] = df_mart['CLASS_ASSESSED_FINAL'].map(severity_map)

severe_classes = ['AA', 'AP NON-IJ']
df_mart['severe_event'] = df_mart['CLASS_ASSESSED_FINAL'].isin(severe_classes).astype(int)
# UPHELD inherits initial class severity
upheld_mask = df_mart['CLASS_ASSESSED_FINAL'] == 'UPHELD'
df_mart.loc[upheld_mask, 'FINAL_SEVERITY'] = (
    df_mart.loc[upheld_mask, 'CLASS_ASSESSED_INITIAL'].map(severity_map)
)

In [ ]:
citations_per_visit = df_mart.groupby(['FACID','EVENTID']).agg(
    facility_name           = ('FACILITY_NAME', 'first'),
    total_action_per_visit  = ('FACILITY_NAME', 'count'),
    location                = ('DISTRICT_OFFICE', 'first'),
    severity_per_action     = ('FINAL_SEVERITY', 'sum'),
    total_deaths            = ('DEATH_RELATED', 'sum'),
    is_low_resource         = ('IS_LOW_RESOURCE', 'first'),
    ltc                     = ('LTC', 'first'),
    appealed                = ('APPEALED', 'sum'),
    balance_due             = ('TOTAL_BALANCE_DUE', 'first'),
    total_offset            = ('TOTAL_PENALTY_OFFSET_AMOUNT', 'first'),
    complaints              = ('COMPLAINT_COUNT', 'sum'),
    last_penalty_date       = ('PENALTY_ISSUE_DATE', 'max'),
).reset_index()

data_end = df_mart['PENALTY_ISSUE_DATE'].max()
citations_per_visit['days_since_last_penalty'] = (
    pd.Timestamp.today() - citations_per_visit['last_penalty_date']
).dt.days

citations_per_visit.head(2)

In [ ]:
facility_summary = citations_per_visit.groupby('FACID').agg(
    facility_name           = ('facility_name', 'first'),
    total_visits            = ('EVENTID', 'count'),
    avg_citations_per_visit = ('total_action_per_visit', 'mean'),
    total_citations         = ('total_action_per_visit', 'sum'),
    total_deaths            = ('total_deaths', 'sum'),
    is_low_resource         = ('is_low_resource', 'first'),
    ltc                     = ('ltc', 'first'),
    total_appealed          = ('appealed', 'sum'),
    total_balance_due       = ('balance_due', 'sum'),
    total_offset            = ('total_offset', 'sum'),
    total_complaints        = ('complaints', 'sum'),
    avg_severity_per_visit  = ('severity_per_action', 'mean'),
    total_severity          = ('severity_per_action', 'sum'),
    days_since_last_penalty = ('days_since_last_penalty', 'min'),  # most recent across all visits
).reset_index()

facility_summary['appeal_rate']       = facility_summary['total_appealed']   / facility_summary['total_citations']
facility_summary['death_rate']        = facility_summary['total_deaths']      / facility_summary['total_citations']
facility_summary['complaint_rate']    = facility_summary['total_complaints']  / facility_summary['total_visits']
facility_summary['balance_per_visit'] = facility_summary['total_balance_due']
facility_summary['risk_recency'] = facility_summary['days_since_last_penalty'].rank(pct=True)

facility_summary.head(2)

In [ ]:
facility_summary['risk_deaths'] = facility_summary['total_deaths'].rank(pct=True)
facility_summary['risk_severity'] = facility_summary['avg_severity_per_visit'].rank(pct=True)
facility_summary['risk_citations'] = facility_summary['avg_citations_per_visit'].rank(pct=True)
facility_summary['risk_complaints'] = facility_summary['total_complaints'].rank(pct=True)
facility_summary['risk_balance'] = facility_summary['total_balance_due'].rank(pct=True)
facility_summary['risk_recency'] = facility_summary['total_balance_due'].rank(pct=True)
citations_per_visit['bad_visit'] = (
    citations_per_visit['severity_per_action'] >
    citations_per_visit['severity_per_action'].quantile(0.75)
).astype(int)

In [ ]:
facility_summary['risk_score'] = (
    facility_summary['risk_deaths'] +
    facility_summary['risk_severity'] +
    facility_summary['risk_balance'] +
    facility_summary['risk_complaints']
)

In [ ]:
top_25 = (
    facility_summary
    .sort_values('risk_score', ascending=False)
    .head(25)[[
        'FACID',
        'facility_name',
        'risk_score',
        'death_rate',
        'avg_severity_per_visit',
        'days_since_last_penalty'
    ]]
)

print(top_25)

past history

In [ ]:
df_mart.columns

In [ ]:
first_severe = (
    df_mart[df_mart['severe_event'] == 1]
    .sort_values('PENALTY_ISSUE_DATE')
    .groupby('FACID')
    .first()
    .reset_index()[['FACID','PENALTY_ISSUE_DATE']]
)

first_severe.rename(columns={'PENALTY_ISSUE_DATE':'first_severe_date'}, inplace=True)

In [ ]:
first_severe

In [ ]:
df_mart = df_mart.merge(first_severe, on='FACID', how='left')

In [ ]:
prior_history = df_mart[
    (df_mart['PENALTY_ISSUE_DATE'] < df_mart['first_severe_date'])
]
prior_citations = (
    prior_history.groupby('FACID')
    .size()
    .reset_index(name='prior_citations')
)
prior_severity = (
    prior_history.groupby('FACID')['FINAL_SEVERITY']
    .sum()
    .reset_index(name='prior_severity')
)
prior_complaints = (
    prior_history.groupby('FACID')['COMPLAINT_COUNT']
    .sum()
    .reset_index(name='prior_complaints')
)

In [ ]:
facility_summary['had_severe_event'] = (
    facility_summary['FACID'].isin(first_severe['FACID'])
).astype(int)
facility_summary.groupby('had_severe_event')[[
    'avg_citations_per_visit',
    'avg_severity_per_visit',
    'total_complaints'
]].mean()

In [ ]:
'PENALTY_ISSUE_DATE'

In [ ]:
time_to_severe = (
    df_mart[df_mart['severe_event'] == 1]
    .groupby('FACID')['PENALTY_ISSUE_DATE']
    .min()
)
time_to_severe

In [ ]:
first_event = df_mart.groupby('FACID')['PENALTY_ISSUE_DATE'].min()

lead_time = (
    first_severe.set_index('FACID')['first_severe_date']
    - first_event
).dt.days
lead_time.describe()

In [ ]:
df_mart['severe_event'].sum()

In [ ]:
first_complaint = (
    df_mart[df_mart['COMPLAINT_COUNT'] > 0]
    .groupby('FACID')['PENALTY_ISSUE_DATE']
    .min()
)

complaint_lead_clean = complaint_lead[complaint_lead >= 0]

complaint_lead_clean.describe()